In [2]:
import h5py
import numpy as np
from pathlib import Path

In [ ]:
#Settings

tag = "LRG_GCcomb_z0.4-0.6_thetacut0.05" #Depends on choice of Desi Data
ells = ["0", "2", "4"]                   #or a subset, order of ells used in calculation



In [ ]:
#Load Data

DATA = Path("Data")
DATA_ELLS = ["0", "2", "4"] #All ells in data

spectrum_file = DATA / ("spectrum-poles-rotated-corrected_" + tag + ".h5")
cov_file = DATA / ("covariance_spectrum-poles-rotated_" + tag + ".h5")
win_file = DATA / ("window_spectrum-poles-rotated_" + tag + ".h5")

#Check

for f in [spectrum_file, cov_file, win_file]:
    print(f.name, f.exists())

In [ ]:
#Get relevant spectrum Data

k, k_edges, P = {}, {}, {}

with h5py.File(spectrum_file) as h:               # step 1: open the file on disk
    for ell in DATA_ELLS:                              # step 2: Run over all ells
        k[ell] = h[f"{ell}/k"][:]                 # step 3: read the dataset "ell/k", ell = 0,2,4
        k_edges[ell] = h[f"{ell}/k_edges"][:]     # step 4: read the dataset "ell/k_edges", ell = 0,2,4
        P[ell] = h[f"{ell}/value"][:]             # step 5: read the dataset "ell/value", ell = 0,2,4

data_full = np.concatenate([P[ell] for ell in DATA_ELLS])   # step 6: Data should be a vector to calculate r = data - model => chi^2
                                                            #         First 36 entries are ell = 0, then ell = 2, then ell = 4
data_mask = make_mask(k_edges)                              # step 7: remove ells not needed for calculation (given by ells from Settings)
data      = data_full[data_mask]  

#Check

for ell in DATA_ELLS:
    print(k_edges[ell].shape)
    print(k[ell].shape)

print(data.shape) 


In [ ]:
#Get relevant covariance Data

cov_edges = {}


with h5py.File(cov_file) as h:
    for ell in DATA_ELLS:
        cov_edges[ell] = h[f"observable/{ell}/k_edges"][:] #Get k_edges for the covariance matrix
    
    C_full = h["value"][:]                                 # Get Covariance Matrix

#Checks

for ell in DATA_ELLS:
    print(cov_edges[ell].shape)

print(C_full.shape)

In [ ]:
#Get relevant window Data

win_edges, k_theory = {}, {}


with h5py.File(win_file) as h:
    for ell in DATA_ELLS:
        win_edges[ell] = h[f"observable/{ell}/k_edges"][:] #Get k_edges for the covariance matrix
        k_theory[ell] = h[f"theory/{ell}/k"][:]
        
    W_full = h["value"][:]                                 # Get Covariance Matrix

#Checks

for ell in DATA_ELLS:
    print(win_edges[ell].shape)
    print(k_theory[ell].shape)

print(W_full.shape)

In [ ]:
#Mask: Cut data to equal size needed

def make_mask(file_edges):                      #file_edges = k_edges, cov_edges, win_edges
    
    mask = []
    
    for ell in ["0", "2", "4"]:
        for e in file_edges[ell]:
            if ell in ells:
                match = np.isclose(e,k_edges[ell]).all(axis=1).any()
            else:
                match = False
            mask.append(match)
            
    return mask